# lulang Embedded: quaternion slerp

Compile one value-semantic numerical kernel, load its generated C ABI with `pylulang`, and compare it with the conventional vectorized NumPy form. Compilation is outside the timed region. The result is checked before timings are reported.

In [ ]:
import os, statistics, sys, time
from pathlib import Path

repo = Path.cwd()
if not (repo / "examples" / "embedded_slerp.lu").exists():
    repo = repo.parent
sys.path.insert(0, str(repo / "python" / "pylulang"))

import numpy as np
import pylulang

compiler = os.environ.get("LULANG_BIN", str(repo / "target" / "release" / "lu"))
module = pylulang.compile(repo / "examples" / "embedded_slerp.lu", name="embedded_slerp", lu=compiler)
count = int(os.environ.get("LULANG_NOTEBOOK_N", "2000000"))
runs = int(os.environ.get("LULANG_NOTEBOOK_RUNS", "5"))

In [ ]:
def numpy_slerp_checksum(n):
    first = np.array([1.0, 2.0, 3.0, 4.0], dtype=np.float64)
    second = np.array([4.0, 3.0, 2.0, 1.0], dtype=np.float64)
    first /= np.linalg.norm(first)
    second /= np.linalg.norm(second)
    t = np.arange(n, dtype=np.float64) / n
    dot = float(first @ second)
    if dot < 0.0:
        dot = -dot
        second = -second
    theta = np.arccos(dot)
    sine = np.sin(theta)
    first_weight = np.sin((1.0 - t) * theta) / sine
    second_weight = np.sin(t * theta) / sine
    quaternions = first_weight[:, None] * first + second_weight[:, None] * second
    quaternions /= np.linalg.norm(quaternions, axis=1)[:, None]
    return float(np.linalg.norm(quaternions, axis=1).sum())

def median_ms(function):
    samples = []
    result = None
    for _ in range(runs):
        started = time.perf_counter_ns()
        result = function()
        samples.append((time.perf_counter_ns() - started) / 1e6)
    return result, statistics.median(samples)

In [ ]:
lulang_result, lulang_ms = median_ms(lambda: module.slerp_checksum(count))
numpy_result, numpy_ms = median_ms(lambda: numpy_slerp_checksum(count))
assert abs(lulang_result - numpy_result) <= 1e-8 * max(1.0, abs(numpy_result))
speedup = numpy_ms / lulang_ms
print(f"checksum: {lulang_result:.12g}")
print(f"lulang: {lulang_ms:.3f} ms")
print(f"NumPy:  {numpy_ms:.3f} ms")
print(f"speedup: {speedup:.2f}x")
assert lulang_ms < numpy_ms, "the Embedded showcase must beat the conventional NumPy implementation on this machine"
module.close()